# 音频清洗：先抽样，再批量

适用于 Google Colab GPU。在第 2 步手动填写 ZIP 的 Google Drive 文件链接或云盘路径。ZIP 中需包含 WAV 音频。流程为 **人声分离 → 轻度降噪 → 对照试听 → 确认后批量处理**。

输出保存到输入 ZIP 所在的云盘文件夹，命名为 `输入文件名_cleaned.zip`。例如 `录音.zip` 对应 `录音_cleaned.zip`。本地工作目录和临时文件也使用 `录音_cleaned` 前缀。分享链接不包含文件名时，会从 Google Drive 读取真实文件名。

先运行到试听步骤，确认效果后再按下面的说明开启批量处理。多人同时说话、强音效或严重削波的样本仍需人工剔除。


In [ ]:
# 1. 安装依赖；Colab 请选择 GPU 运行时
!pip -q install 'audio-separator[gpu]' 'noisereduce>=3,<4' 'soundfile>=0.12,<1' 'scipy>=1.11,<2'
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
assert gpu.returncode == 0, '请在 Colab「运行时 → 更改运行时类型」选择 GPU'
print('GPU:', gpu.stdout.strip())


In [ ]:
#@title 2. 填写待清洗 ZIP 的 Google Drive 地址
# 支持文件分享链接、/content/drive/MyDrive/.../录音.zip 或 MyDrive/.../录音.zip
INPUT_DRIVE_ADDRESS = "" #@param {type:"string"}

from pathlib import Path
from urllib.parse import urlparse, parse_qs, unquote
from google.colab import drive, auth
import re, zipfile, shutil, hashlib

def configure_paths(input_name):
    assert input_name.lower().endswith('.zip'), '输入文件应为 ZIP'
    assert '/' not in input_name and '\\' not in input_name, '文件名不能包含目录分隔符'
    stem = Path(input_name).stem
    assert stem not in ('', '.', '..'), '输入文件名无效'
    prefix = stem + '_cleaned'
    base = Path('/content') / prefix
    return {
        'PREFIX': prefix, 'BASE': base,
        'RAW': base / (prefix + '_raw'),
        'PILOT': base / (prefix + '_pilot'),
        'CLEAN': base / (prefix + '_wavs'),
        'STEMS': base / (prefix + '_stems'),
        'ZIP': base / (prefix + '_input.zip'),
        'DOWNLOAD_TMP': base / (prefix + '_input.zip.tmp'),
        'LOCAL_ZIP': base / (prefix + '.zip'),
        'MANIFEST_PATH': base / (prefix + '_manifest.json'),
        'OUTPUT_NAME': prefix + '.zip',
    }

def md5_file(path):
    digest = hashlib.md5()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

address = INPUT_DRIVE_ADDRESS.strip() or input('请输入待清洗 ZIP 的 Google Drive 链接或路径：').strip()
assert address, '请填写待清洗 ZIP 的 Google Drive 地址'
IS_DRIVE_LINK = address.startswith(('https://', 'http://'))
if IS_DRIVE_LINK:
    parsed = urlparse(address)
    assert parsed.hostname == 'drive.google.com', '请填写 drive.google.com 的 ZIP 文件链接'
    query = parse_qs(parsed.query)
    match = re.search(r'/file/d/([A-Za-z0-9_-]+)', parsed.path)
    file_id = match.group(1) if match else query.get('id', [''])[0]
    assert re.fullmatch(r'[A-Za-z0-9_-]+', file_id), '请复制 ZIP 文件分享链接，不要使用文件夹链接'
    auth.authenticate_user()
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
    credentials, _ = google.auth.default()
    service = build('drive', 'v3', credentials=credentials, cache_discovery=False)
    resource_key = query.get('resourcekey', [''])[0]
    def with_resource_key(request):
        if resource_key:
            request.headers['X-Goog-Drive-Resource-Keys'] = f'{file_id}/{resource_key}'
        return request
    metadata = with_resource_key(service.files().get(
        fileId=file_id, fields='name,parents', supportsAllDrives=True)).execute()
    INPUT_NAME = metadata['name']
    parents = metadata.get('parents', [])
    assert parents, '无法确定输入 ZIP 所在的云盘文件夹；请改用挂载后的文件路径'
    OUTPUT_FOLDER_ID = parents[0]
    folder = service.files().get(fileId=OUTPUT_FOLDER_ID,
        fields='capabilities(canAddChildren)', supportsAllDrives=True).execute()
    assert folder.get('capabilities', {}).get('canAddChildren'), '当前账号无法向输入 ZIP 所在文件夹写入结果'
    OUTPUT_LOCATION = f'https://drive.google.com/drive/folders/{OUTPUT_FOLDER_ID}'
else:
    drive.mount('/content/drive')
    source_path = Path(unquote(address))
    if not source_path.is_absolute(): source_path = Path('/content/drive') / source_path
    source_path = source_path.resolve()
    assert source_path.is_relative_to(Path('/content/drive')), '路径应为 /content/drive/... 或 MyDrive/...'
    assert source_path.is_file(), f'云盘文件不存在：{source_path}'
    INPUT_NAME = source_path.name
    STORE = source_path.parent
    OUTPUT_LOCATION = str(STORE)

globals().update(configure_paths(INPUT_NAME))
for folder in (BASE, RAW, PILOT, CLEAN): folder.mkdir(parents=True, exist_ok=True)
if IS_DRIVE_LINK:
    request = with_resource_key(service.files().get_media(fileId=file_id, supportsAllDrives=True))
    with DOWNLOAD_TMP.open('wb') as stream:
        downloader = MediaIoBaseDownload(stream, request, chunksize=16 * 1024 * 1024)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status: print(f'读取云盘 ZIP：{status.progress():.0%}', flush=True)
else:
    shutil.copy2(source_path, DOWNLOAD_TMP)

assert zipfile.is_zipfile(DOWNLOAD_TMP), '该地址的文件不是有效 ZIP'
DOWNLOAD_TMP.replace(ZIP)
with zipfile.ZipFile(ZIP) as archive:
    entries = [e for e in archive.infolist() if not e.is_dir() and e.filename.lower().endswith('.wav')]
    names = [Path(e.filename).name for e in entries]
    assert entries and len(names) == len(set(names)), 'ZIP 中没有 WAV，或存在同名 WAV'
    for old in RAW.glob('*.wav'): old.unlink()
    for entry, name in zip(entries, names):
        with archive.open(entry) as src, (RAW / name).open('wb') as dst:
            shutil.copyfileobj(src, dst)
sources = sorted(RAW.glob('*.wav'))

def save_to_drive(local_path, mime_type):
    # 路径输入写回同目录；链接输入通过 Drive API 写回原文件夹。
    if not IS_DRIVE_LINK:
        assert STORE.is_dir(), '云盘目录不可用，请重新挂载 Google Drive'
        target = STORE / local_path.name
        temporary = STORE / (local_path.name + '.tmp')
        shutil.copy2(local_path, temporary)
        assert md5_file(temporary) == md5_file(local_path), '云盘文件完整性校验失败'
        temporary.replace(target)
        return str(target)
    escaped_name = local_path.name.replace('\\', '\\\\').replace("'", "\\'")
    existing = service.files().list(
        q=f"'{OUTPUT_FOLDER_ID}' in parents and name = '{escaped_name}' and trashed = false",
        fields='files(id),nextPageToken', pageSize=100,
        supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
    matches = existing.get('files', [])
    assert len(matches) <= 1 and not existing.get('nextPageToken'), '云盘目录存在多个同名输出，请先整理同名文件'
    media = MediaFileUpload(str(local_path), mimetype=mime_type, resumable=True)
    options = dict(media_body=media, fields='id,size,md5Checksum', supportsAllDrives=True)
    if matches:
        request = service.files().update(fileId=matches[0]['id'], **options)
    else:
        request = service.files().create(body={'name': local_path.name, 'parents': [OUTPUT_FOLDER_ID]}, **options)
    result = None
    while result is None:
        status, result = request.next_chunk()
        if status: print(f'保存 {local_path.name}：{status.progress():.0%}', flush=True)
    assert int(result['size']) == local_path.stat().st_size, '云盘文件大小校验失败'
    assert result.get('md5Checksum') == md5_file(local_path), '云盘文件完整性校验失败'
    return f"https://drive.google.com/file/d/{result['id']}/view"

print('输入 ZIP：', INPUT_NAME)
print(f'找到 {len(sources)} 条 WAV；工作目录：{BASE}')
print('云盘输出目录：', OUTPUT_LOCATION)
print('云盘输出文件名：', OUTPUT_NAME)


In [ ]:
# 3. 参数：先保持默认值试听，必要时只调整 DENOISE_STRENGTH 后重跑试清理
MODEL = 'model_bs_roformer_ep_317_sdr_12.9755.ckpt'
DENOISE_STRENGTH = 0.55  # 0=跳过降噪，1=最强；建议先试 0.4~0.65
PILOT_COUNT = 5
MANUAL_PILOT_NAMES = []  # 可填 ['clip_00000.wav', ...]，覆盖自动抽样
RUN_BATCH = False         # 听完并确认后，手动改成 True
assert 0 <= DENOISE_STRENGTH <= 1


In [ ]:
# 4. 准备分离器和清理函数。模型首次下载需要一些时间。
import gc, json
import numpy as np
import soundfile as sf
import noisereduce as nr
from audio_separator.separator import Separator

STEMS.mkdir(exist_ok=True)
separator = Separator(output_dir=str(STEMS), output_format='WAV', sample_rate=44100)
separator.load_model(model_filename=MODEL)

def describe(path):
    info = sf.info(path)
    return {'name': path.name, 'seconds': round(info.duration, 2), 'sample_rate': info.samplerate, 'channels': info.channels}

def clean_one(src, destination):
    # 每个文件独立分离；只接受唯一的人声 stem，避免误取伴奏。
    for old in STEMS.glob('*'): old.unlink()
    outputs = separator.separate(str(src))
    vocals = [STEMS / Path(p).name for p in outputs if 'vocal' in Path(p).stem.lower()]
    assert len(vocals) == 1 and vocals[0].is_file(), f'无法确定人声输出：{src.name} {outputs}'
    y, sr = sf.read(vocals[0], dtype='float32', always_2d=True)
    if DENOISE_STRENGTH:
        y = np.stack([nr.reduce_noise(y=y[:, ch], sr=sr, stationary=False,
                                      prop_decrease=DENOISE_STRENGTH) for ch in range(y.shape[1])], axis=1)
    y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
    peak = float(np.max(np.abs(y))) if y.size else 0.0
    if peak > 0.98: y *= 0.98 / peak
    destination.parent.mkdir(parents=True, exist_ok=True)
    sf.write(destination, y, sr, subtype='PCM_16')
    for stem in vocals: stem.unlink(missing_ok=True)
    gc.collect()
    return {'name': src.name, 'seconds': round(len(y) / sr, 2), 'peak': round(peak, 3)}


In [ ]:
# 5. 抽样：自动覆盖短、中、长时长；也可在上一格指定文件名
ordered = sorted(sources, key=lambda p: sf.info(p).duration)
if MANUAL_PILOT_NAMES:
    pilot_sources = [RAW / name for name in MANUAL_PILOT_NAMES]
    assert all(p.is_file() for p in pilot_sources), 'MANUAL_PILOT_NAMES 含不存在的文件'
else:
    positions = np.linspace(0, len(ordered) - 1, min(PILOT_COUNT, len(ordered)), dtype=int)
    pilot_sources = [ordered[i] for i in dict.fromkeys(positions)]
print('试清理样本：', [describe(p) for p in pilot_sources])
pilot_results = []
for i, src in enumerate(pilot_sources, 1):
    print(f'试清理 {i}/{len(pilot_sources)}: {src.name}', flush=True)
    pilot_results.append(clean_one(src, PILOT / src.name))
print('完成：', pilot_results)


In [ ]:
# 6. 逐条试听。留意尾音、气声、齿音是否被削弱，以及背景音乐是否仍明显。
from IPython.display import Audio, display
for src in pilot_sources:
    print('原音:', src.name)
    display(Audio(filename=str(src)))
    print('清理:', src.name)
    display(Audio(filename=str(PILOT / src.name)))


## 确认后再批量

试听满意后，按以下顺序操作：

1. 往上找到 **第 3 步“参数”** 的代码单元格，将 `RUN_BATCH = False` 改成 `RUN_BATCH = True`。
2. 点击运行 **第 3 步**，让修改生效。
3. 再运行下面的 **第 7 步“批量处理并导出”**，开始清理全部音频，并将 ZIP 保存到 Google Drive。

第 7 步中的 `assert RUN_BATCH is True` 保持原样。

若人声发闷、尾音消失，降低 `DENOISE_STRENGTH`；若仍有底噪，适度提高。调整降噪强度后，先运行第 3 步，再运行第 5 步试清理和第 6 步试听，确认效果后再批量处理。分离无法可靠去除另一位说话者的声音，遇到此类样本请从训练集剔除。


In [ ]:
# 7. 批量处理并导出；默认关闭。重跑会更新同名结果文件。
assert RUN_BATCH is True, '请先试听，再在第 3 步将 RUN_BATCH 改为 True 并运行第 3 步'
if not IS_DRIVE_LINK:
    assert STORE.is_dir(), '云盘目录不可用，请重新运行第 2 步'
for old in CLEAN.glob('*.wav'): old.unlink()
records = []
for i, src in enumerate(sources, 1):
    print(f'批量 {i}/{len(sources)}: {src.name}', flush=True)
    records.append(clean_one(src, CLEAN / src.name))
assert len(list(CLEAN.glob('*.wav'))) == len(sources), '输出数量与输入不一致'
manifest = {'input_name': INPUT_NAME, 'output_name': OUTPUT_NAME, 'model': MODEL,
            'denoise_strength': DENOISE_STRENGTH, 'input_count': len(sources), 'files': records}
MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
with zipfile.ZipFile(LOCAL_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for wav in sorted(CLEAN.glob('*.wav')): archive.write(wav, wav.name)
    archive.write(MANIFEST_PATH, MANIFEST_PATH.name)
with zipfile.ZipFile(LOCAL_ZIP) as check:
    assert check.testzip() is None, '本地 ZIP 完整性校验失败'
OUT_ZIP = save_to_drive(LOCAL_ZIP, 'application/zip')
save_to_drive(MANIFEST_PATH, 'application/json')
print('已保存并校验 Google Drive：', OUT_ZIP)


## 接入训练

在训练 notebook 中填写生成的 `输入文件名_cleaned.zip` 的云盘路径或链接，并关闭重复的人声分离。ZIP 内的 WAV 保留原文件名，便于对照与标注。清理后仍建议在 ASR 校对阶段试听并剔除串音或失真的片段。
